# 5일차 실습 — ADMET 예측과 결과 해석

이 노트북은 **Claude로 진행하는 실습의 대체용**이다.
Claude에서 코드 실행이 되지 않거나 `pip install`이 막힌 경우에 이 노트북을 쓴다.

실습의 목적은 예측값을 얻는 것이 아니라 **그 값을 쓸 수 있는지 판단하는 것**이다.
각 절의 마지막에 있는 「직접 확인」 과제를 반드시 채운다.

| 절 | 하는 일 | Claude 프롬프트 |
|---|---|---|
| 1 | 설치와 예측 | ① |
| 2 | 값을 의심하기 — 기준 화합물 10개 | ②~⑥ |
| 3 | 생성분자에 적용 | — |
| 4 | 자체 데이터 파인튜닝 | ⑦ |
| 5 | 보고 표 만들기 | ⑧ |
| 부록 | MPO 스코어링 (선택) | — |

> 런타임은 CPU로 충분하다. GPU가 있으면 4절이 빨라진다.


---
## 1. 설치와 예측

**Claude 프롬프트 ①**
> admet-ai 패키지를 설치하고, 아래 10개 SMILES에 대해 ADMET을 예측해서 표로 정리해 줘.
> 회귀 항목과 분류 항목을 구분하고, 각 항목의 단위를 함께 적어 줘.


### 1-1. 설치

`torch`와 `chemprop`을 함께 받으므로 수 분 걸린다. 설치를 걸어 두고 다음 설명을 읽는다.

> **버전을 고정하는 이유** — ADMET-AI는 PyPI의 최신이 `1.4.0`이고, 이 실습이 쓰는 `2.0.1`은
> GitHub 태그(`v_2.0.1`)에만 있다. 2.0.0에서 **모델이 chemprop v1 → v2로 전부 교체**되어
> 두 버전은 같은 분자에 다른 값을 낸다. `pip install admet-ai`로 받으면 강의 화면의 숫자가
> 재현되지 않고, 4절(파인튜닝)은 chemprop v2 API를 쓰므로 아예 실행되지 않는다.

> **Python 3.11 이상이 필요하다** — chemprop 2.x의 요구사항이다. Colab은 조건을 만족한다.


In [ ]:
# ADMET-AI 2.0.1 — GitHub 태그에서 받는다 (PyPI 최신은 1.4.0이라 값이 다르다)
!pip -q install "git+https://github.com/swansonk14/admet_ai.git@v_2.0.1" rdkit pandas matplotlib


In [ ]:
# 설치 확인 — 이 세 줄이 아래와 맞아야 이후 결과가 강의 화면과 같다
import sys, importlib.metadata as md
print("python  ", sys.version.split()[0], "  (3.11 이상)")
print("admet_ai", md.version("admet_ai"), "  (2.0.1)")
print("chemprop", md.version("chemprop"), "  (2.x)")


### 1-2. 입력 — 기준 화합물 10개

정답이 알려진 화합물로 시작한다. **참값을 알아야 예측이 맞았는지 판단할 수 있기 때문이다.**

| 화합물 | 왜 넣었는가 |
|---|---|
| atenolol · metoprolol | FDA BCS 가이던스의 투과도 기준 화합물 (low / high) |
| verapamil · terfenadine · amiodarone | hERG를 차단하는 것으로 알려져 있다 |
| digoxin · amiodarone | 분포용적이 매우 크다 |
| warfarin | 혈장 단백 결합이 높다 |
| aspirin · caffeine · propranolol | 물성이 잘 알려진 대조군 |


In [ ]:
import pandas as pd

REF = {
    "aspirin":     "CC(=O)Oc1ccccc1C(=O)O",
    "caffeine":    "Cn1cnc2c1c(=O)n(C)c(=O)n2C",
    "atenolol":    "CC(C)NCC(O)COc1ccc(CC(N)=O)cc1",
    "metoprolol":  "COCCc1ccc(OCC(O)CNC(C)C)cc1",
    "propranolol": "CC(C)NCC(O)COc1cccc2ccccc12",
    "verapamil":   "COc1ccc(CCN(C)CCCC(C#N)(C(C)C)c2ccc(OC)c(OC)c2)cc1OC",
    "amiodarone":  "CCCCc1oc2ccccc2c1C(=O)c1cc(I)c(OCCN(CC)CC)c(I)c1",
    "digoxin":     "CC1OC(CC(O)C1O)OC1CC(OC2CC(O)C(OC3CC(O)C(O)C(C)O3)C(C)O2)C(C)OC1OC1CCC2(C)C(CCC3C2CCC2(C)C(C4=CC(=O)OC4)CCC32O)C1",
    "warfarin":    "CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O",
    "terfenadine": "CC(C)(C)c1ccc(C(O)CCCN2CCC(C(O)(c3ccccc3)c3ccccc3)CC2)cc1",
}
print(len(REF), '개 화합물')


### 1-3. 예측

모델을 한 번만 만들고 `predict`를 호출한다. 첫 실행은 가중치를 읽느라 시간이 걸린다.


In [ ]:
from admet_ai import ADMETModel

model = ADMETModel()
pred  = model.predict(smiles=list(REF.values()))
pred.index = list(REF.keys())
print('출력 형태:', pred.shape, '(분자 x 열)')
print('예측값 열:', len([c for c in pred.columns if 'percentile' not in c]))
print('백분위 열:', len([c for c in pred.columns if 'percentile' in c]))


> **직접 확인 ①** 열이 104개나 되는 이유는 무엇인가?
> 예측값 열과 백분위 열이 어떤 관계인지 한 문장으로 적는다.

```
답: 
```


---
## 2. 값을 의심하기 ← **이 실습의 핵심**

여기서부터는 Claude에게 물어보고 **그 답을 검증**하는 구간이다.
노트북으로 할 때도 같은 순서로 진행한다.


### 2-1. 단위를 먼저 본다

**Claude 프롬프트 ⑥**
> 이 화합물들의 용해도를 좋은 순서로 정렬해 줘.

`Solubility_AqSolDB`의 단위는 **log mol/L**이다. 값이 **클수록** 잘 녹는다.
단위를 확인하지 않으면 방향을 반대로 읽는다.


In [ ]:
cols = ['Caco2_Wang', 'Solubility_AqSolDB', 'Lipophilicity_AstraZeneca',
        'PPBR_AZ', 'VDss_Lombardo', 'Half_Life_Obach',
        'Clearance_Microsome_AZ', 'LD50_Zhu']

units = {'Caco2_Wang': 'log(10^-6 cm/s)', 'Solubility_AqSolDB': 'log(mol/L)',
         'Lipophilicity_AstraZeneca': 'logD7.4', 'PPBR_AZ': '% bound',
         'VDss_Lombardo': 'L/kg', 'Half_Life_Obach': 'h',
         'Clearance_Microsome_AZ': 'uL/min/mg', 'LD50_Zhu': 'log(1/(mol/kg))'}

reg = pred[cols].round(2)
reg.columns = [f'{c} [{units[c]}]' for c in cols]
reg


> **직접 확인 ②** 위 표에서 용해도가 가장 좋은 화합물과 가장 나쁜 화합물은?
> `LD50_Zhu`는 값이 클수록 독한가, 덜 독한가?

```
용해도 최고 / 최저: 
LD50 방향: 
```


### 2-2. 물리적으로 불가능한 값 찾기

**Claude 프롬프트 ②**
> 이 표에서 물리적으로 불가능한 값이 있는지 찾아 줘.


In [ ]:
phys = ['VDss_Lombardo', 'Half_Life_Obach',
        'Clearance_Microsome_AZ', 'Clearance_Hepatocyte_AZ', 'PPBR_AZ']
bad  = pred[phys].round(2)

def mark(v):
    return 'background-color: #FBF1E6; color: #B45309; font-weight: bold' if v < 0 else ''

print('음수가 나온 칸의 개수:', int((bad < 0).sum().sum()))
print(bad.to_string())
try:
    disp = bad.style.map(mark)          # pandas 2.1 이상
except AttributeError:
    disp = bad.style.applymap(mark)     # 구버전
disp


> **직접 확인 ③** 왜 이 값들이 불가능한가? 각 항목의 정의로 설명한다.

```
Vdss: 
Half-life: 
Clearance: 
```


### 2-3. 근거를 도구 안에서 찾는다

**Claude 프롬프트 ③**
> 왜 그런 값이 나오는지 패키지 안에서 근거를 찾아 줘.

ADMET-AI는 **자기 성능표를 패키지에 함께 넣어 배포한다.** 직접 열어 본다.


In [ ]:
import os, admet_ai

D  = os.path.dirname(admet_ai.__file__)
ad = pd.read_csv(os.path.join(D, 'resources/data/admet.csv'))

perf = ad[(ad.task_type == 'regression') & ad['R^2'].notna()]
perf[['id', 'size', 'units', 'minimum', 'maximum', 'R^2', 'MAE']].round(3)


**읽는 법**

- `R^2`가 **음수**면 그 모델의 예측이 **학습셋 평균을 찍는 것보다 오차가 크다**는 뜻이다.
- `minimum` 열에 0.0이 적혀 있다 — 도구는 물리적 하한을 알고 있으면서 출력에 적용하지 않는다.
- 이 세 데이터셋은 **로그 변환 없이 원 단위로 배포**된다. 분포가 오른쪽으로 길게 늘어진 상태에서
  제곱오차를 최소화하면 예측이 평균 쪽으로 눌리고, 하한 제약이 없으면 음수까지 내려간다.

> **직접 확인 ④** R²가 음수인 항목 두 개를 적고, 그 값이 얼마인지 쓴다.
> 그리고 「이 항목의 예측값을 보고서에 쓸 것인가」에 답한다.

```
항목 / R²: 
판단: 
```


### 2-4. 알려진 사실과 대조한다

**Claude 프롬프트 ④**
> atenolol과 metoprolol의 투과도 예측 순서가 알려진 사실과 맞는지 확인해 줘.

FDA BCS 가이던스는 **metoprolol을 high permeability, atenolol을 low permeability**로 분류한다.
예측이 이 순서를 재현하는지 본다.


In [ ]:
chk = pred.loc[['atenolol', 'metoprolol'], ['Caco2_Wang', 'PAMPA_NCATS', 'HIA_Hou']].round(3)
# Caco2_Wang을 log10(Papp / (cm/s))로 보면 아래와 같이 환산된다
chk['Papp (10^-6 cm/s)'] = (10 ** pred.loc[['atenolol', 'metoprolol'], 'Caco2_Wang'] * 1e6).round(1)
chk


**단위 표기가 어긋나 있다**

ADMET-AI의 메타데이터는 `Caco2_Wang`의 단위를 `log(10^-6 cm/s)`로 적고 있고,
TDC 문서는 `cm/s`로 적고 있다. 배포된 라벨의 실제 값은 −7 ~ −4 범위의 음수이므로
**log10(Papp / (cm/s))로 보는 것이 값의 범위와 맞는다.** 위 환산은 그 해석을 따랐다.

> 공개 도구의 단위 표기가 서로 어긋나는 일은 드물지 않다.
> 값을 인용하기 전에 **분포의 범위로 단위를 역산해 확인**하는 습관이 필요하다.


> **직접 확인 ⑤** 순서가 맞았는가? `HIA_Hou`는 두 화합물을 구분했는가?
> 구분하지 못했다면 그 이유는 무엇이라고 생각하는가?

```
답: 
```


### 2-5. 확률은 무엇에 대한 확률인가

**Claude 프롬프트 ⑤**
> verapamil의 hERG 예측이 0.97인데, 이 화합물을 위험하다고 판정할 수 있나?

**Claude가 「위험하다」로 단정하면 그것이 오답이다.** 근거는 아래 두 가지다.

1. `hERG` 모델은 **hERG_Karim 데이터셋의 라벨 기준(IC₅₀ < 10 µM)**으로 학습되었다.
   0.97은 「그 기준에서 blocker로 분류될 확률」이지 위험도가 아니다.
2. 안전성 판정은 **hERG IC₅₀와 유리 치료 혈장농도 사이의 안전역**으로 한다
   (Redfern WS et al., *Cardiovasc. Res.* 58(1):32, 2003 — 30배 이상 권고).
   **verapamil은 안전역이 1.7배에 불과하지만 TdP를 일으키지 않는다** — 다중 이온채널을 함께 차단하기 때문이다.


In [ ]:
herg = pred[['hERG']].round(3)
herg['hERG 백분위'] = pred['hERG_drugbank_approved_percentile'].round(1)
herg.sort_values('hERG', ascending=False)


> **직접 확인 ⑥** Claude의 답을 아래에 옮기고, 위 두 근거로 검증한다.

```
Claude의 답: 
내 검증: 
판정 (맞음 / 틀림 / 확인 불가): 
```


### 2-6. 검증 기록표

여기까지의 결과를 한 표로 정리한다. **이 표가 2절의 산출물이다.**


In [ ]:
record = pd.DataFrame({
    '프롬프트': ['② 불가능한 값', '③ 근거 추적', '④ 기준 화합물',
               '⑤ hERG 해석', '⑥ 용해도 정렬'],
    'Claude의 답': [''] * 5,
    '내 검증':     [''] * 5,
    '판정':       [''] * 5,
})
record  # 직접 채운 뒤 record.to_csv('검증기록.csv', index=False) 로 저장


---
## 3. 생성분자에 적용

기준 화합물로 도구를 **교정**했으니, 이제 참값을 모르는 분자에 쓴다.
4일차에서 생성한 분자를 이어받는다.


In [ ]:
import os, subprocess
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

ASSETS = "https://github.com/eunjoolee122/2026-aidrugdiscovery/releases/download/assets-v1/assets.zip"

if not os.path.exists("trxr_generated.csv"):
    subprocess.run(["wget", "-q", "-O", "assets.zip", ASSETS])
    subprocess.run(["unzip", "-o", "-q", "-j", "assets.zip",
                    "content/assets/trxr_generated.csv"])

raw = pd.read_csv("trxr_generated.csv")["SMILES"].dropna().tolist()

def canon(s):
    m = Chem.MolFromSmiles(str(s))
    return Chem.MolToSmiles(m) if m else None

gen = list(dict.fromkeys(c for s in raw if (c := canon(s))))[:200]
print(f"생성분자 {len(raw)}개 → 유효·중복제거 후 상위 {len(gen)}개로 진행")


In [ ]:
gen_pred = model.predict(smiles=gen)
print('예측 완료:', gen_pred.shape)
gen_pred[['QED', 'logP', 'tpsa', 'Caco2_Wang', 'hERG', 'AMES', 'DILI']].head()


### 3-1. 쓸 수 있는 항목만 남긴다

2절에서 확인한 대로, R²가 음수인 항목은 값으로 쓰지 않는다.


In [ ]:
DROP = ['VDss_Lombardo', 'Half_Life_Obach']          # R^2 < 0
WEAK = ['Clearance_Hepatocyte_AZ', 'Clearance_Microsome_AZ']  # R^2 ~ 0.27

usable = [c for c in gen_pred.columns
          if 'percentile' not in c and c not in DROP + WEAK]
print(f'전체 예측 항목에서 {len(DROP)}개 제외, {len(WEAK)}개는 주의 표시')
print('남은 항목 수:', len(usable))


### 3-2. 절대값이 아니라 백분위로 읽는다

보정되지 않은 회귀값은 승인약 분포에서의 **상대 위치**로 바꾸어 읽는다.

> 그래프에 한글을 쓰려면 폰트가 필요하다. 아래 셀이 없으면 한글이 사각형으로 표시된다.


In [ ]:
# 한글 폰트 — Colab에서 한 번만 실행하면 된다 (없으면 영문 라벨로 자동 대체)
import matplotlib, matplotlib.pyplot as plt
from matplotlib import font_manager

KO = None
for name in ['NanumGothic', 'NanumBarunGothic', 'Malgun Gothic', 'AppleGothic']:
    if any(name in f.name for f in font_manager.fontManager.ttflist):
        KO = name; break
if KO is None:
    try:
        import subprocess
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'fonts-nanum'], check=True)
        font_manager.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
        KO = 'NanumGothic'
    except Exception:
        pass
if KO:
    matplotlib.rcParams['font.family'] = KO
    matplotlib.rcParams['axes.unicode_minus'] = False
print('그래프 한글 폰트:', KO or '없음 — 영문 라벨을 씁니다')


In [ ]:
import matplotlib.pyplot as plt

PCT = ['Caco2_Wang', 'Solubility_AqSolDB', 'hERG', 'DILI', 'AMES']
pct = gen_pred[[f'{c}_drugbank_approved_percentile' for c in PCT]]
pct.columns = PCT

fig, ax = plt.subplots(figsize=(9, 4))
pct.boxplot(ax=ax)
ax.axhline(50, color='gray', ls='--', lw=1)
ax.set_ylabel('승인약 대비 백분위' if KO else 'percentile vs approved drugs')
ax.set_title('생성분자 200개 — 승인약 분포에서의 위치' if KO
             else 'generated molecules (n=200) vs approved drugs')
plt.tight_layout(); plt.show()


> **직접 확인 ⑦** 생성분자는 승인약 대비 어느 항목에서 유리하고 어느 항목에서 불리한가?
> 중앙선(50)을 기준으로 읽는다.

```
답: 
```


---
## 4. 자체 데이터로 파인튜닝

**Claude 프롬프트 ⑦**
> 첨부한 `er_dataset.csv`는 자체적으로만 측정한 endpoint라고 하자.
> ADMET-AI는 이 항목을 예측하지 않는다. 공개 모델의 encoder를 재사용해서 예측 모델을 만들고,
> 처음부터 학습시킨 경우와 성능을 비교해 줘.

**데이터** — Biogen이 공개한 ADME 세트의 MDR1-MDCK log efflux ratio 2,642개
(단일 실험실 · 동일 조건 · CC BY 4.0). ADMET-AI가 예측하지 않는 항목이므로
**자체 전용 endpoint의 대역**으로 쓴다.

> Fang C et al., *J. Chem. Inf. Model.* **63**:3263 (2023)


In [ ]:
import numpy as np, torch

URL = "https://raw.githubusercontent.com/molecularinformatics/Computational-ADME/main/ADME_public_set_3521.csv"
src = pd.read_csv(URL)
col = "LOG MDR1-MDCK ER (B-A/A-B)"
er  = src[["SMILES", col]].dropna().rename(columns={"SMILES": "smiles", col: "log_er"})
print(f"efflux ratio 데이터 {len(er)}개")
er.describe().round(3)


### 4-1. ADMET-AI 체크포인트에서 encoder만 가져온다

패키지의 `.pt` 파일은 **Chemprop 2.x의 MPNN 체크포인트**다. 두 부분으로 되어 있다.

| 구성 요소 | 역할 | 파라미터 |
|---|---|---|
| `message_passing` (BondMessagePassing, d_h=300) | 분자 그래프 → 표현 벡터 300차원. **재사용 대상** | 227,700 (71%) |
| `predictor` (RegressionFFN, 2층) | 표현 → 10개 회귀 과제. **교체 대상** | 93,310 (29%) |

자체 항목은 ADMET-AI의 41개 과제에 없으므로 `predictor`는 세 방식 모두 **새로 만들어 학습한다.**
세 방식의 차이는 `message_passing`을 어떻게 다루는가뿐이다.

| 방식 | encoder 초기값 | encoder 학습 | 학습되는 파라미터 |
|---|---|---|---|
| 처음부터 학습 | 무작위 | 학습 | 321,010 (100%) |
| encoder 고정 | ADMET-AI | **고정** — 역전파가 여기까지 내려오지 않는다 | 93,310 (29%) |
| encoder 파인튜닝 | ADMET-AI | 이어 학습 | 321,010 (100%) |

「고정」은 `requires_grad_(False)`로 구현한다. encoder가 매번 같은 값을 내놓는 고정된 함수가 되므로
학습 시간이 짧아진다.

> 체크포인트가 macOS에서 저장되어 있어 `map_location="cpu"`를 지정해야 한다.


In [ ]:
from lightning import pytorch as pl
from chemprop import data, models, nn, featurizers
from sklearn.metrics import r2_score, mean_absolute_error

torch.manual_seed(0); np.random.seed(0)

pts = [data.MoleculeDatapoint.from_smi(s, np.array([y]))
       for s, y in zip(er.smiles, er.log_er)]
tr, va, te = data.make_split_indices(np.array([p.mol for p in pts]), "random", (0.8, 0.1, 0.1), seed=0)
trd, vad, ted = data.split_data_by_indices(pts, tr, va, te)
fz = featurizers.SimpleMoleculeMolGraphFeaturizer()
DT = data.MoleculeDataset(trd[0], fz)
DV = data.MoleculeDataset(vad[0], fz)
DE = data.MoleculeDataset(ted[0], fz)
sc = DT.normalize_targets(); DV.normalize_targets(sc)
print('train / val / test =', len(DT), len(DV), len(DE))


In [ ]:
CK = os.path.join(D, 'resources/models/admet_regression/model_0.pt')

def build(pretrained, freeze):
    if pretrained:
        mp = models.MPNN.load_from_file(CK, map_location='cpu').message_passing
        if freeze:
            mp.apply(lambda m: m.requires_grad_(False)); mp.eval()
    else:
        mp = nn.BondMessagePassing(d_h=300)
    ffn = nn.RegressionFFN(input_dim=300,
                           output_transform=nn.UnscaleTransform.from_standard_scaler(sc))
    return models.MPNN(mp, nn.MeanAggregation(), ffn, batch_norm=True)

y_true = np.array([d.y[0] for d in DE.data])
rows = []

for tag, pre, frz in [('처음부터 학습', False, False),
                      ('ADMET-AI encoder 고정', True, True),
                      ('ADMET-AI encoder 파인튜닝', True, False)]:
    m = build(pre, frz)
    t = pl.Trainer(max_epochs=25, accelerator='auto', logger=False,
                   enable_progress_bar=False, enable_checkpointing=False,
                   enable_model_summary=False)
    t.fit(m, data.build_dataloader(DT, num_workers=0),
          data.build_dataloader(DV, num_workers=0, shuffle=False))
    p = np.concatenate([x.numpy() for x in
                        t.predict(m, data.build_dataloader(DE, num_workers=0, shuffle=False))]).ravel()
    rows.append((tag, r2_score(y_true, p), mean_absolute_error(y_true, p)))
    print(f'{tag:26s} R2 = {rows[-1][1]:6.3f}  MAE = {rows[-1][2]:.3f}', flush=True)

pd.DataFrame(rows, columns=['방식', 'R2', 'MAE']).round(3)


> **분류 항목에 적용할 때** — `RegressionFFN` 대신 `nn.BinaryClassificationFFN`을 쓰고,
> 체크포인트도 `resources/models/admet_classification/model_0.pt`를 불러온다.
> `input_dim=300`은 encoder의 출력 차원이며, RDKit descriptor를 함께 넣으려면 그만큼 늘린다.


**미리 확인한 결과 — 시드 3개로 반복 (CPU, 25 epoch)**

| 방식 | R² 평균 | R² 범위 | MAE 평균 | 학습 시간 |
|---|---|---|---|---|
| 처음부터 학습 | **0.519** | 0.447 ~ 0.564 | 0.367 | 161초 |
| ADMET-AI encoder 파인튜닝 | 0.507 | 0.450 ~ 0.538 | 0.376 | 154초 |
| ADMET-AI encoder 고정 | 0.432 | 0.375 ~ 0.491 | 0.425 | **95초** |

**한 번의 실행으로는 방법을 고를 수 없다.**

- 처음부터 학습과 파인튜닝의 차이(0.519 대 0.507)는 **시드 간 편차(각각 0.12 · 0.09)보다 작다.**
  세 시드 중 두 번은 오히려 처음부터 학습이 앞섰다.
- 첫 실행에서는 파인튜닝 0.460, 처음부터 학습 0.407로 파인튜닝이 분명히 좋아 보였다.
  그 차이는 반복하니 사라졌다.
- **encoder를 고정한 경우만 일관되게 낮다.** 대신 학습 시간이 40% 짧다 —
  예측할 항목이 많을 때는 이 이점이 크다.

> 전이학습이 무용하다는 뜻은 아니다. 2,642개 규모에서는 이득이 시드 편차에 묻힌다는 뜻이다.

> **직접 확인 ⑧** 위 셀을 seed를 바꿔 두 번 더 돌려 본다.
> 어느 방식이 이기는가? 같은 결과가 나오는가?
> 「이 실험 결과로 자체 어느 방식을 권하겠는가」에 답한다.

```
1회차 (seed=0): 
2회차 (seed=?): 
판단: 
```


### 4-2. 시드를 바꿔 반복한다

위 셀의 `seed=0`을 바꾸어 다시 돌린다. 분할과 초기화가 함께 바뀐다.


In [ ]:
# seed만 바꿔 같은 비교를 반복한다 (한 번에 약 7~8분)
def compare(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    pts = [data.MoleculeDatapoint.from_smi(s, np.array([y]))
           for s, y in zip(er.smiles, er.log_er)]
    tr, va, te = data.make_split_indices(np.array([p.mol for p in pts]),
                                         "random", (0.8, 0.1, 0.1), seed=seed)
    trd, vad, ted = data.split_data_by_indices(pts, tr, va, te)
    DT = data.MoleculeDataset(trd[0], fz)
    DV = data.MoleculeDataset(vad[0], fz)
    DE = data.MoleculeDataset(ted[0], fz)
    sc2 = DT.normalize_targets(); DV.normalize_targets(sc2)
    y = np.array([d.y[0] for d in DE.data])
    out = []
    for tag, pre, frz in [('처음부터', False, False), ('고정', True, True), ('파인튜닝', True, False)]:
        torch.manual_seed(seed)
        if pre:
            mp = models.MPNN.load_from_file(CK, map_location='cpu').message_passing
            if frz: mp.apply(lambda m: m.requires_grad_(False)); mp.eval()
        else:
            mp = nn.BondMessagePassing(d_h=300)
        ffn = nn.RegressionFFN(input_dim=300,
                               output_transform=nn.UnscaleTransform.from_standard_scaler(sc2))
        m = models.MPNN(mp, nn.MeanAggregation(), ffn, batch_norm=True)
        t = pl.Trainer(max_epochs=25, accelerator='auto', logger=False,
                       enable_progress_bar=False, enable_checkpointing=False,
                       enable_model_summary=False)
        t.fit(m, data.build_dataloader(DT, num_workers=0),
              data.build_dataloader(DV, num_workers=0, shuffle=False))
        p = np.concatenate([x.numpy() for x in t.predict(
            m, data.build_dataloader(DE, num_workers=0, shuffle=False))]).ravel()
        out.append((seed, tag, r2_score(y, p)))
        print(f'seed {seed} {tag:6s} R2={out[-1][2]:.3f}', flush=True)
    return out

# 시간이 없으면 한 시드만 추가로 돌려 본다
extra = compare(1)
pd.DataFrame(extra, columns=['seed', '방식', 'R2']).round(3)


---
## 5. 보고 표 만들기

**Claude 프롬프트 ⑧**
> 오늘 결과를 팀에 보고할 표로 정리해 줘.
> 도구 이름과 버전 · 실행 날짜 · 항목의 원 단위 · 분류 항목의 라벨 기준 ·
> 승인약 대비 백분위 · 값을 쓸 수 없는 항목과 그 이유를 반드시 포함할 것.


In [ ]:
import admet_ai, datetime

report = pd.DataFrame([
    ['도구',            f'ADMET-AI {admet_ai.__version__ if hasattr(admet_ai, "__version__") else "2.0.1"}'],
    ['실행 날짜',        datetime.date.today().isoformat()],
    ['입력 분자 수',     len(gen)],
    ['제외한 항목',      'VDss_Lombardo (R2 -1.21), Half_Life_Obach (R2 -2.39)'],
    ['주의 표시 항목',   'Clearance_* (R2 0.26~0.28)'],
    ['분류 항목 라벨 기준', 'hERG: IC50 < 10 uM (hERG_Karim) / CYP*: AC50 <= 10 uM (Veith)'],
    ['해석 기준',        '절대값이 아니라 승인약 대비 백분위로 읽음'],
    ['판정',            '후보 선별용으로만 사용. 판정은 실측으로 한다'],
], columns=['항목', '내용'])
report


> **제출물** — 2절의 검증 기록표 + 위 보고 표


---
## 부록 (선택) — MPO 스코어링

시간이 남으면 진행한다. 여러 물성을 **하나의 점수로 합치는** 방법이다.

| 단계 | 하는 일 | 예 |
|---|---|---|
| ① 원시값 | 분자에서 물성을 뽑는다 | MW = 474.6, logP = 1.61 |
| ② transform | 원시값을 0~1 점수로 바꾼다 | MW 474.6 → 0.912 |
| ③ 결합 | 점수들을 하나로 합친다 | 기하평균 → 0.726 |

`double_sigmoid`는 [low, high] 범위 안일수록 1에 가까운 종 모양 함수다.
`geometric_mean`은 **하나라도 0점이면 전체가 0점**이 되어, 모든 조건을 동시에 만족하라는 압력을 만든다.
구조경보는 가중평균에 들어가지 않고 **최종 점수에 곱해지는 필터**다.

> 아래 구현은 REINVENT4의 scoring 결과와 소수점 7자리까지 일치하도록 맞춘 것이다.


In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, QED

ALERT_SMARTS = ["[*;r{8-17}]", "[#8][#8]", "[#6;+]", "[#16][#16]", "C#C"]
_ALERTS = [Chem.MolFromSmarts(s) for s in ALERT_SMARTS]

def _sigmoid(x, k):
    h = np.clip(k * np.asarray(x, float) * np.log(10), -700, 700)
    return np.where(h >= 0, 1 / (1 + np.exp(-h)), np.exp(h) / (1 + np.exp(h)))

def double_sigmoid(x, low, high, coef_div, coef_si, coef_se):
    x = np.asarray(x, float)
    center = (high - low) / 2 + low
    out = np.zeros_like(x)
    left = x < center
    out[left]  = _sigmoid(x[left] - low, coef_si / coef_div)
    out[~left] = 1 - _sigmoid(x[~left] - high, coef_se / coef_div)
    return out

def geometric_mean(scores, weights):
    s = np.maximum(np.array(scores, float), 1e-8)
    w = np.array(weights, float); w = w / w.sum()
    return np.prod(s ** w[:, None], axis=0)

mols     = [Chem.MolFromSmiles(s) for s in gen]
raw_qed  = np.array([QED.qed(m) for m in mols])
raw_mw   = np.array([Descriptors.MolWt(m) for m in mols])
raw_logp = np.array([Crippen.MolLogP(m) for m in mols])

s_qed   = raw_qed
s_mw    = double_sigmoid(raw_mw, 200, 500, 500, 20, 20)
s_logp  = double_sigmoid(raw_logp, 1, 5, 5, 20, 20)
s_alert = np.array([0.0 if any(m.HasSubstructMatch(p) for p in _ALERTS) else 1.0
                    for m in mols])

mpo = geometric_mean([s_qed, s_mw, s_logp], [0.5, 0.25, 0.25]) * s_alert

out = pd.DataFrame({'SMILES': gen, 'QED': raw_qed.round(3), 'MW': raw_mw.round(1),
                    'logP': raw_logp.round(2), 'alert': s_alert.astype(int),
                    'MPO': mpo.round(4)})
out.sort_values('MPO', ascending=False).head(10)


> **직접 확인 ⑨** MPO 점수가 0인 분자가 있는가? 그 이유는?
> 산술평균을 썼다면 결과가 어떻게 달라지겠는가?

```
답: 
```


---
## 정리

1. 예측값은 **단위를 확인한 뒤에** 읽는다. 로그인지 아닌지에 따라 방향이 바뀐다.
2. 분류 확률은 **그 데이터셋의 라벨 기준에 대한 확률**이다. 모델의 신뢰도도, 위험도도 아니다.
3. 도구가 배포하는 성능표를 먼저 본다. **R²가 음수인 항목은 값으로 쓰지 않는다.**
4. 정답이 알려진 화합물로 도구를 교정한 뒤, 모르는 분자에 쓴다.
5. 보고할 때 도구 · 버전 · 날짜 · 단위 · 라벨 기준을 함께 적는다.
